# DealScout - Phase 3, Step 3: ScannerAgent & MessagingAgent

Scan RSS deal feeds for promising products, and send push notifications via pushover

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import logging
import requests
from dealscout.agents.deals import ScrapedDeal, DealSelection

load_dotenv(override=True)
openai = OpenAI()
MODEL = 'gpt-5-mini'

In [4]:
deals = ScrapedDeal.fetch(show_progress=True)

100%|██████████| 3/3 [00:46<00:00, 15.65s/it]


In [5]:
len(deals)

30

In [6]:
deals[10].describe()

'Title: Samsung Monitors at Woot: Up to 48% off + extra 20% off + free shipping w/ Prime\nDetails: This Woot sale features new Samsung monitors at up to 48% off reference prices, with an extra 20% off in the Woot app (25% for new customers, up to $50 discount). Highlights include the Samsung 32" Odyssey G5 QHD Monitor at $200, down from $300, the Samsung 32" Smart Monitor M8 4K UHD Display at $450, down from $700, and the Samsung 57" DUHD Monitor at $1,450, down from $1,800. The app offer ends today. Shop Now at Woot! An Amazon Company\nFeatures: 13 monitor models available Sizes range from 27" to 57" Resolutions include FHD, QHD, 4K UHD, and DUHD Panel types include OLED, curved, and flat App discount: up to 25% extra off (max $50 for new customers)\nURL: https://www.dealnews.com/Samsung-Monitors-at-Woot-Up-to-48-off-extra-20-off-free-shipping-w-Prime/21846465.html?iref=rss-c39'

In [7]:
SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 
"""

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""

USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."

In [8]:
def make_user_prompt(scraped):
    user_prompt = USER_PROMPT_PREFIX
    user_prompt += '\n\n'.join([scrape.describe() for scrape in scraped])
    user_prompt += USER_PROMPT_SUFFIX
    return user_prompt

In [9]:
user_prompt = make_user_prompt(deals)
print(user_prompt[:2000])
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}]

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: Like-New Amazon Devices Sale at Woot for From $20 + extra 20% off + free shipping w/ Prime
Details: Make your purchase via the Woot app to bag the extra 20% off. Shop for like-new refurbished Amazon devices without power adapters, starting at $19.99 for the Echo Auto, $24.99 for the Echo Dot (2022), and $39.99 for the Fire HD 10 Tablet (2021). All items are backed by a 90-day W

In [10]:
response = openai.chat.completions.parse(model=MODEL, messages=messages, response_format=DealSelection, reasoning_effort="minimal")
results = response.choices[0].message.parsed
results

DealSelection(deals=[Deal(product_description='Refurbished like-new Amazon devices including Echo Auto, Echo Dot (2022), and Fire HD 10 Tablet (2021) sold without power adapters or accessories. Each unit is inspected and graded like-new and comes with a 90‑day Woot limited warranty. Prime members get free standard shipping and additional app-only discounts are available when purchased through the Woot app.', price=19.99, url='https://www.dealnews.com/Like-New-Amazon-Devices-Sale-at-Woot-for-From-20-extra-20-off-free-shipping-w-Prime/21846464.html?iref=rss-c142'), Deal(product_description='Amazon Echo Dot bundled with an officially licensed Star Wars stand (Mandalorian helmet or Grogu). The Echo Dot provides voice control for music, smart home devices, and Alexa features; the themed stand secures the 4th- and 5th-gen Dot without blocking microphones or speakers and the Mandalorian helmet illuminates when Alexa is activated. The bundle includes the Echo Dot (Charcoal) and collectible the

In [11]:
for deal in results.deals:
    print(deal.product_description)
    print(deal.price)
    print(deal.url)
    print()


Refurbished like-new Amazon devices including Echo Auto, Echo Dot (2022), and Fire HD 10 Tablet (2021) sold without power adapters or accessories. Each unit is inspected and graded like-new and comes with a 90‑day Woot limited warranty. Prime members get free standard shipping and additional app-only discounts are available when purchased through the Woot app.
19.99
https://www.dealnews.com/Like-New-Amazon-Devices-Sale-at-Woot-for-From-20-extra-20-off-free-shipping-w-Prime/21846464.html?iref=rss-c142

Amazon Echo Dot bundled with an officially licensed Star Wars stand (Mandalorian helmet or Grogu). The Echo Dot provides voice control for music, smart home devices, and Alexa features; the themed stand secures the 4th- and 5th-gen Dot without blocking microphones or speakers and the Mandalorian helmet illuminates when Alexa is activated. The bundle includes the Echo Dot (Charcoal) and collectible themed stand.
80.0
https://www.dealnews.com/Echo-Dot-Star-Wars-Bundle-for-Mando-or-Grogu-for

In [12]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [14]:
from dealscout.agents.scanner_agent import ScannerAgent

In [15]:
agent = ScannerAgent()
result = agent.scan()

INFO:root:[Scanner Agent] Scanner Agent is initializing
INFO:root:[Scanner Agent] Scanner Agent is ready
INFO:root:[Scanner Agent] Scanner Agent is about to fetch deals from RSS feed
INFO:root:[Scanner Agent] Scanner Agent received 30 deals not already scraped
INFO:root:[Scanner Agent] Scanner Agent is calling OpenAI using Structured Outputs
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:[Scanner Agent] Scanner Agent received 5 selected deals with price>0 from OpenAI


In [16]:
result

DealSelection(deals=[Deal(product_description='The Aoostar Maco 6850H is a compact mini PC built around the AMD Ryzen 7 Pro 6850H CPU, offering multi-core performance for productivity and light content creation. It ships with 24GB of LPDDR5 system memory, Wi‑Fi 6, a 2.5G LAN port for fast wired networking, and modern connectivity including USB4, HDMI, and DisplayPort outputs. The unit is sold without an internal SSD, making it suitable for users who want to choose their own storage configuration in a small-footprint, VESA-mountable chassis.', price=266.0, url='https://www.dealnews.com/Aoostar-Maco-6850-H-AMD-Ryzen-7-Pro-Mini-PC-for-266-free-shipping/21844645.html?iref=rss-c39'), Deal(product_description='The Magcubic L018 is a portable projector with a native 1080p panel and 650 ANSI lumen brightness for bright home theater projection. It runs Android 14, supports 8K content playback, and includes Wi‑Fi 6 and Bluetooth 5.4 for streaming and accessory connections. Hardware conveniences 

In [3]:
load_dotenv(override=True)

True

In [4]:
pushover_user = os.getenv('PUSHOVER_USER')
pushover_token = os.getenv('PUSHOVER_TOKEN')
pushover_url = "https://api.pushover.net/1/messages.json"

In [5]:
if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

Pushover user found and starts with u
Pushover token found and starts with a


In [6]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [7]:
push("MASSIVE DEAL!!")

Push: MASSIVE DEAL!!


In [2]:
from dealscout.agents.messaging_agent import MessagingAgent

agent = MessagingAgent()
agent.push("SUCH A MASSIVE DEAL!!")

In [3]:
agent.notify("A special deal on Sumsung 60 inch LED TV going at a great bargain", 300, 1000, "www.samsung.com")